# Analysis v2: intervals for the final results, error analysis, paper figures (CPU)

No GPU, no training, no new prediction, and no data split is opened for evaluation:
1. `quickdraw_exp.bootstrap`: 95% bootstrap intervals for every final_v1 comparison, from the predictions that
   final_eval saved for later analyses (`predictions_test/`, `predictions_unseen_countries/`) and the validation
   calibration halves. Thresholds are rebuilt exactly as in final_v1; the notebook stops if any rebuilt number
   differs from `reports/final_v1_test/results.json`.
2. `quickdraw_exp.errors`: early confusions and stopping by class, on validation.
3. `paper/make_assets.py`: the two figures that need the drawings (validation only).

**Before running:**
1. **Input**: your Kaggle Dataset with the processed data; `ai-pictionary-step4a-output` (version 4, holds
   `runs/predictions_all/`); and the dataset made from **AI_pictionary version 10** output (final_v1, holds
   `runs/final_v1_test/predictions_test/`). The notebook stops if one is missing.
2. **Accelerator: None** (CPU). Internet on (for the code).
3. **Save Version → Save & Run All**.


In [ ]:
import os, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Trung4n/AI-dictionary.git"
BRANCH = "pipeline-step1"
REPO = Path("/kaggle/working/repo")

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    print("No GITHUB_TOKEN secret: cloning without one (works for a public repository).")

url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
os.chdir("/kaggle/working")  # a re-run may still be inside the folder deleted next
if REPO.exists():
    shutil.rmtree(REPO)
done = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(REPO)], capture_output=True, text=True)
if done.returncode != 0:
    message = done.stderr.replace(token, "***") if token else done.stderr
    raise RuntimeError(f"clone failed:\n{message}")
# Keep the token out of files that Kaggle may save with the notebook output.
subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL], check=True)
print(subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"], capture_output=True, text=True).stdout)


In [ ]:
%cd /kaggle/working/repo
!pip install -q pytest
tests = subprocess.run("PYTHONPATH=src python -m pytest tests -q -k 'not processed_data'", shell=True, capture_output=True, text=True)
print(tests.stdout[-1500:])
if tests.returncode != 0:
    raise RuntimeError("tests failed: stopping")

import numpy as np

def find_folder(file_name, parent_name):
    for p in Path("/kaggle/input").rglob(file_name):
        if p.parent.name == parent_name:
            return p.parent
    return None

ALL = find_folder("cnn_prefix_s42.npz", "predictions_all")
TEST_PRED = find_folder("gru_uniform_s42.npz", "predictions_test")
if ALL is None:
    raise RuntimeError("runs/predictions_all not found: attach ai-pictionary-step4a-output (version 4)")
if TEST_PRED is None:
    raise RuntimeError("predictions_test not found: attach the dataset made from AI_pictionary version 10 (final_v1)")
FINAL = TEST_PRED.parent
for name in ("predictions_test", "predictions_unseen_countries"):
    print(name, sorted(p.name for p in (FINAL / name).glob("*.npz")))
print("validation predictions:", ALL)


In [ ]:
!PYTHONPATH=src python -m quickdraw_exp.bootstrap --val-predictions {ALL} --final-predictions {FINAL} --out /kaggle/working/runs/bootstrap_v1


In [ ]:
!PYTHONPATH=src python -m quickdraw_exp.errors --predictions {ALL} --out /kaggle/working/runs/errors_v1


In [ ]:
# The two paper figures that need the drawings (validation only); the tables are rebuilt too.
!python paper/make_assets.py --val-predictions {ALL}
FIGS = Path("/kaggle/working/paper_figures")
FIGS.mkdir(exist_ok=True)
for name in ("fig_prefix_example.pdf", "fig_qualitative.pdf"):
    source = Path("paper/figures") / name
    if source.is_file():
        shutil.copy(source, FIGS / name)
print(sorted(p.name for p in FIGS.iterdir()))
